# 003 · Decimal Arithmetic

> **Standard Library › Numbers & Math** · Python 3.14

Python's `float` is a binary IEEE-754 double, just like C#'s `double` — so `0.1 + 0.2 != 0.3`. For money and anything else that must match human (base-10) arithmetic, use `decimal.Decimal`. It plays the role of C#'s `decimal`, but with **configurable precision** and explicit control over rounding and error handling.

**In this notebook you will learn:**

- how `Decimal` compares to C#'s `decimal`
- why you must construct a `Decimal` from a **string**, not a float
- contexts: precision, `localcontext`, and why precision means *significant digits*
- rounding money with `quantize` and `ROUND_HALF_UP` vs `ROUND_HALF_EVEN`
- signals and traps (`DivisionByZero`, `Inexact`, ...)
- the "no mixing with float" rule

## .NET mental model

| | C# `decimal` | Python `Decimal` |
|---|---|---|
| Literal | `19.99m` | `Decimal("19.99")` (no literal syntax) |
| Precision | fixed: 28–29 significant digits, 128 bits | **configurable**, default 28 significant digits, can be thousands |
| Range | about ±7.9 × 10²⁸ | exponent up to ±999 999 by default |
| Trailing zeros kept | yes (`1.30m` keeps scale 2) | yes (`Decimal("1.30")`) |
| Default rounding (`Math.Round` / `round`) | `MidpointRounding.ToEven` | `ROUND_HALF_EVEN` |
| Divide by zero | `DivideByZeroException` | `decimal.DivisionByZero` (configurable) |
| Mix with `double`/`float` | compile error without a cast | `TypeError` at runtime |
| Namespace | built-in type `System.Decimal` | `from decimal import Decimal` |

Both are **decimal floating-point** types: they store a base-10 coefficient and an exponent, so `0.1` is exact.

In [ ]:
from decimal import Decimal

print(0.1 + 0.2)                               # float: 0.30000000000000004
print(Decimal("0.1") + Decimal("0.2"))         # exactly 0.3 — like 0.1m + 0.2m
print(Decimal("1.30") + Decimal("1.20"))       # 2.50 — scale is preserved
print(Decimal("1.30") * Decimal("1.20"))       # 1.5600

## Construct from `str`, not `float`

⚠️ **Gotcha:** this is the #1 `Decimal` mistake. `Decimal(0.1)` converts the float **exactly** — and the float `0.1` is not exactly 0.1. You get every binary-rounding artefact, faithfully preserved.

In C#, converting a `double` to `decimal` rounds it to about 15 significant digits behind your back, which hides the problem; Python doesn't.

In [ ]:
print(Decimal(0.1))       # the true value of the double 0.1
print(Decimal("0.1"))     # what you meant
print(Decimal(str(0.1)))  # if you only have a float, go via str()

Integers are always safe, and so are tuples or other `Decimal`s. Invalid strings raise `InvalidOperation` (not `ValueError` as you might guess):

In [ ]:
from decimal import InvalidOperation

print(Decimal(42), Decimal("-1.5e3"), Decimal("  7.25  "))   # whitespace is stripped
try:
    Decimal("12,50")
except InvalidOperation as e:
    print(f"{type(e).__name__}: {e}")

## Contexts and precision

Every arithmetic operation runs under a **context** that controls precision, rounding mode and which conditions raise. `getcontext()` returns the current thread's context (and the current asyncio task's — it is context-local).

In [ ]:
from decimal import getcontext

ctx = getcontext()
print(ctx.prec, ctx.rounding)                                 # 28 ROUND_HALF_EVEN
print("traps:", [s.__name__ for s, on in ctx.traps.items() if on])
print(Decimal(1) / Decimal(3))                                # 28 significant digits

⚠️ **Gotcha:** `prec` is the number of **significant digits** of a result, not the number of decimal places. With `prec=5`, a large number loses its fraction *and* some integer digits:

In [ ]:
from decimal import localcontext

with localcontext(prec=5):                     # temporary change, restored on exit
    print(Decimal(1) / Decimal(3))              # 0.33333
    print(Decimal("123456.789") + 0)            # 1.2346E+5 — only 5 significant digits

Unlike C#'s fixed 28–29 digits, you can go much higher when you need it:

In [ ]:
big = Decimal(10) ** 30
tiny = Decimal("0.000001")
print(big + tiny)                               # default 28 digits: the tiny part is lost

with localcontext(prec=50):
    print(big + tiny)                           # exact with 50 digits
    print(Decimal(2).sqrt())

**Tip:** prefer `with localcontext(...)` over mutating `getcontext()` directly — changes to the global context leak into every other piece of code running in the same thread. Note that precision applies to **results of arithmetic**; constructing a `Decimal` from a string is always exact.

## Rounding money with `quantize`

`quantize(exp)` rounds a value to the **same exponent** as `exp` — so `Decimal("0.01")` means "two decimal places". This is the standard way to round currency.

```csharp
decimal tax = Math.Round(subtotal * rate, 2, MidpointRounding.AwayFromZero);
```

In [ ]:
from decimal import ROUND_HALF_UP, ROUND_HALF_EVEN

CENTS = Decimal("0.01")

price, qty, rate = Decimal("19.99"), 3, Decimal("0.0825")
subtotal = price * qty                    # Decimal * int is fine
tax = subtotal * rate
print(subtotal, tax)                      # 59.97 4.947525
print(tax.quantize(CENTS, rounding=ROUND_HALF_UP))   # 4.95

### `ROUND_HALF_UP` vs `ROUND_HALF_EVEN`

The two midpoint rules you'll meet most often:

| Python | C# `MidpointRounding` | 2.5 → | 3.5 → | Typical use |
|---|---|---|---|---|
| `ROUND_HALF_EVEN` (context default) | `ToEven` (default) | 2 | 4 | "banker's rounding", avoids bias in sums |
| `ROUND_HALF_UP` | `AwayFromZero` | 3 | 4 | what people learn at school; many invoices/tax rules |

Despite its name, Python's `ROUND_HALF_UP` rounds **away from zero** (−2.5 → −3), matching `AwayFromZero`.

In [ ]:
for s in ["0.125", "0.135", "-0.125"]:
    d = Decimal(s)
    print(f"{s:>7}  half-even: {d.quantize(CENTS, rounding=ROUND_HALF_EVEN):>6}"
          f"  half-up: {d.quantize(CENTS, rounding=ROUND_HALF_UP):>6}")

Other modes exist too: `ROUND_DOWN` (toward zero, like `MidpointRounding.ToZero` / `Math.Truncate`), `ROUND_FLOOR`, `ROUND_CEILING`, `ROUND_UP` (away from zero), `ROUND_HALF_DOWN`, `ROUND_05UP`.

In [ ]:
from decimal import ROUND_DOWN, ROUND_CEILING

print(Decimal("7.329").quantize(CENTS, rounding=ROUND_DOWN))      # 7.32
print(Decimal("7.321").quantize(CENTS, rounding=ROUND_CEILING))   # 7.33

⚠️ **Gotcha:** built-in `round(d, 2)` on a `Decimal` works and returns a `Decimal`, but it always uses the context's rounding (half-even by default). Use `quantize` when the rounding rule matters — it makes the rule explicit.

In [ ]:
print(round(Decimal("2.345"), 2), Decimal("2.345").quantize(CENTS, rounding=ROUND_HALF_UP))

## Signals and traps

Exceptional conditions are **signals**. For each signal the context has:

- a **flag** that is set when the condition occurs (sticky, until cleared)
- a **trap** that, if enabled, turns the signal into an exception

By default `InvalidOperation`, `DivisionByZero` and `Overflow` are trapped (raise); `Inexact`, `Rounded` etc. only set flags. All decimal exceptions derive from `ArithmeticError`.

In [ ]:
from decimal import DivisionByZero

try:
    Decimal(1) / Decimal(0)
except DivisionByZero as e:
    print(f"{type(e).__name__}: {e}")

with localcontext() as ctx:
    ctx.traps[DivisionByZero] = False      # disable the trap...
    print(Decimal(1) / Decimal(0))          # ...and get Infinity instead

Enabling the `Inexact` trap is a neat way to **guarantee** a calculation lost nothing — useful in financial code where silent rounding is unacceptable:

In [ ]:
from decimal import Inexact

with localcontext() as ctx:
    ctx.traps[Inexact] = True
    print(Decimal(1) / Decimal(4))          # 0.25 is exact — fine
    try:
        Decimal(1) / Decimal(3)             # 0.333... must be rounded
    except Inexact as e:
        print(f"{type(e).__name__}: {e}")

Or leave traps off and inspect the flags afterwards:

In [ ]:
from decimal import Rounded

with localcontext() as ctx:
    ctx.clear_flags()
    Decimal(10) / Decimal(4)
    print("after 10/4:", ctx.flags[Inexact])
    Decimal(10) / Decimal(3)
    print("after 10/3:", ctx.flags[Inexact], ctx.flags[Rounded])

## No mixing with `float`

⚠️ **Gotcha:** arithmetic between `Decimal` and `float` raises `TypeError` — the same protection C# gives you at compile time. `int` mixes fine.

In [ ]:
amount = Decimal("10.50")
print(amount * 2, amount + 1)              # int: OK

try:
    amount * 1.1
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print(amount * Decimal("1.1"))             # convert explicitly instead

Comparisons *are* allowed, and they compare **exact** values — so a float that isn't exactly representable never equals the "same" decimal:

In [ ]:
print(Decimal("0.5") == 0.5)     # True — 0.5 is exact in binary
print(Decimal("1.1") == 1.1)     # False — the float is 1.100000000000000088...
print(Decimal("1.10") == Decimal("1.1"))   # True — trailing zeros don't affect equality

## Formatting and converting

`Decimal` supports the usual format specs *(see Syntax › Data Types › 003 · String Formatting)*. Convert out with `int()` (truncates) or `float()` (may lose precision).

In [ ]:
total = Decimal("1234567.5")
print(f"{total:,.2f}")                     # 1,234,567.50
print(int(Decimal("9.99")), float(Decimal("0.1")))
print(Decimal("123.4500").normalize())     # strip trailing zeros: 123.45

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Literal | `19.99m` | `Decimal("19.99")` |
| From a double | `(decimal)d` | `Decimal(str(f))` — never `Decimal(f)` |
| Round to cents, half-up | `Math.Round(x, 2, MidpointRounding.AwayFromZero)` | `x.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)` |
| Banker's rounding | `Math.Round(x, 2)` | `x.quantize(Decimal("0.01"))` (half-even default) |
| Truncate | `Math.Truncate(x)` | `x.quantize(Decimal("1"), rounding=ROUND_DOWN)` / `int(x)` |
| More precision | not possible (fixed) | `with localcontext(prec=50):` |
| Catch divide by zero | `catch (DivideByZeroException)` | `except decimal.DivisionByZero` |
| Detect any rounding | — | trap or check `decimal.Inexact` |
| Mix with double | compile error | `TypeError` |
| Format | `x.ToString("N2")` | `f"{x:,.2f}"` |

## Exercises

1. Add up the prices `["0.10", "0.20"]` as `float`s and as `Decimal`s. Show that only the decimal total compares equal to `0.30` / `Decimal("0.30")`.
2. Write `split_bill(total: Decimal, people: int) -> list[Decimal]` that splits a bill into cent amounts which add up **exactly** to the total (the first few people pay one extra cent if needed). Test with `Decimal("100.00")` and 3 people.
3. Round `["2.345", "2.355", "-2.345"]` to 2 places with both `ROUND_HALF_UP` and `ROUND_HALF_EVEN` and print a small table.
4. Compute `1 / 7` with 60 significant digits without affecting the global context.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from decimal import Decimal, ROUND_DOWN, ROUND_HALF_UP, ROUND_HALF_EVEN, localcontext

# 1
prices = ["0.10", "0.20"]
print(sum(float(p) for p in prices) == 0.30)                      # False
print(sum(Decimal(p) for p in prices) == Decimal("0.30"))         # True

# 2 — round everyone down to the cent, then hand out the leftover cents
def split_bill(total: Decimal, people: int) -> list[Decimal]:
    cent = Decimal("0.01")
    share = (total / people).quantize(cent, rounding=ROUND_DOWN)
    leftover_cents = int((total - share * people) / cent)
    return [share + cent if i < leftover_cents else share for i in range(people)]

parts = split_bill(Decimal("100.00"), 3)
print(parts, sum(parts))          # [Decimal('33.34'), Decimal('33.33'), Decimal('33.33')] 100.00

# 3
cents = Decimal("0.01")
for s in ["2.345", "2.355", "-2.345"]:
    d = Decimal(s)
    print(s, d.quantize(cents, ROUND_HALF_UP), d.quantize(cents, ROUND_HALF_EVEN))

# 4
with localcontext(prec=60):
    print(Decimal(1) / Decimal(7))
```

</details>